# 02 — Mapper basics

`Mapper` is the scarlets SDK's distributed key-value store: any number of
workers `Map` a value under their own key, independently and without
coordinating with each other, and any node can `AllGather` everything
that's been written so far. `Reduce` folds the gathered values with an
operator; `resetAll`/`clearAll` overwrite or delete everything.

Fixed names + a cleanup cell up front make this notebook safe to re-run
from the top.

In [ ]:
import os
import numpy as np
from scarlets.core.Mapper import Mapper
from scarlets.utils.ScarletUtils import redisConnect

os.environ.setdefault("REDIS_HOST", "localhost")
os.environ.setdefault("REDIS_PORT", "6379")
os.environ.setdefault("REDIS_AUTH_TOKEN", "")
os.environ.setdefault("APP_ID", "notebook_worker")

MAPPER_NAME = "tutorial_mapper"

## Cleanup

In [ ]:
def cleanup():
    r = redisConnect()
    for pattern in (f"{MAPPER_NAME}_key-value:*", f"{MAPPER_NAME}_key-list"):
        keys = list(r.scan_iter(match=pattern))
        if keys:
            r.delete(*keys)

cleanup()
print("cleaned up")

## Map — independent writes

Three simulated workers each `Map` their own reading under their own key.
They don't need to know about each other, or how many other workers exist.

In [ ]:
mpr = Mapper(MAPPER_NAME)

for i, value in enumerate([10.0, 20.0, 30.0]):
    chunks, ok, exc = mpr.Map(np.array([value]), f"worker_{i}")
    print(f"worker_{i}: ok={ok}")

## AllGather — reading everything back

Any Mapper instance pointed at the same name sees every key written so far,
regardless of which worker wrote it.

In [ ]:
result, ok, exc = mpr.AllGather()
result

## Reduce — fold across all values

`Reduce` calls `AllGather` internally, then folds every value into a
running result with the given operator. `Mapper.SUM`/`MAX`/`MIN`/`MUL` are
built in.

In [ ]:
total, ok, exc = mpr.Reduce(np.zeros(1), op=Mapper.SUM)
total

## resetAll — overwrite every existing key

Every key currently registered gets overwritten with the same value -
useful for e.g. resetting all workers to a shared starting point.

In [ ]:
mpr.resetAll(np.array([0.0]))
mpr.AllGather()

## clearAll — delete everything

Removes every key this Mapper currently tracks.

In [ ]:
mpr.clearAll()
result, ok, exc = mpr.AllGather()
print(f"keys remaining: {len(result)}")

## Cleanup (teardown)

In [ ]:
cleanup()
print("cleaned up")